# FID Evaluation and Model Comparison

This notebook demonstrates how to use Fréchet Inception Distance (FID) to evaluate and compare GAN models trained with different hyperparameters.

**FID Score Interpretation:**
- FID measures the distance between the distribution of real and generated images
- **Lower FID is better** - it indicates higher quality and diversity of generated images
- Typical FID ranges: 
  - Excellent: < 10
  - Good: 10-20
  - Fair: 20-50
  - Poor: > 50

## 1. Import Required Libraries

In [1]:
import torch
import torch.nn as nn
import numpy as np
from scipy.linalg import sqrtm
from torchvision.models import inception_v3
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import os
from glob import glob
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
import sys

# Add the project root to the path so we can import our FID module
project_root = Path.cwd()
sys.path.append(str(project_root))

# Import our custom FID evaluation module
from fid_evaluation import compute_fid, evaluate_checkpoint, evaluate_all_checkpoints, compare_hyperparameters

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Load Real Images and Define Generator Classes

In [2]:
import os
from PIL import Image

# Load real image paths
base_path = "data/villagers" 
image_paths = []

for root, _, files in os.walk(base_path):
    for f in files:
        if f.lower().endswith(".png"):
            image_paths.append(os.path.join(root, f))

print(f"Found {len(image_paths)} real images")

# Create a simple dataset class for real images
class ImageDataset(Dataset):
    def __init__(self, image_paths, max_size=None):
        self.image_paths = image_paths[:max_size] if max_size else image_paths
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path)
        
        # Handle transparency
        if img.mode == 'RGBA':
            bg = Image.new('RGB', img.size, (255, 255, 255))
            bg.paste(img, mask=img.split()[3])
            img = bg
        else:
            img = img.convert('RGB')
        
        # Convert to tensor and normalize to [-1, 1]
        img_array = np.array(img).astype(np.float32) / 127.5 - 1.0
        img_tensor = torch.from_numpy(img_array).permute(2, 0, 1)
        
        return img_tensor

Found 392 real images


In [3]:
# Define Generator classes (copy from your DCGAN.ipynb and WGAN-GP.ipynb)

class DCGANGenerator(nn.Module):
    """DCGAN Generator architecture"""
    def __init__(self, latent_dim=10, img_channels=3):
        super(DCGANGenerator, self).__init__()
        self.latent_dim = latent_dim
        
        self.linear = nn.Linear(latent_dim, 256 * 4 * 4)
        
        self.main = nn.Sequential(
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.ConvTranspose2d(64, img_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )
    
    def forward(self, z):
        x = self.linear(z)
        x = x.view(-1, 256, 4, 4)
        x = self.main(x)
        return x

class WGANGPGenerator(nn.Module):
    """WGAN-GP Generator architecture"""
    def __init__(self, latent_dim=10, img_channels=3):
        super(WGANGPGenerator, self).__init__()
        self.latent_dim = latent_dim
        
        self.linear = nn.Linear(latent_dim, 256 * 4 * 4)
        
        self.main = nn.Sequential(
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.ConvTranspose2d(64, img_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )
    
    def forward(self, z):
        x = self.linear(z)
        x = x.view(-1, 256, 4, 4)
        x = self.main(x)
        return x

print("Generator classes defined")

Generator classes defined


## 3. Calculate FID Score for All Checkpoints

In [4]:
# Define checkpoint directories
checkpoint_dirs = {
    'DCGAN': 'checkpoints/dcgan',
    'WGAN-GP': 'checkpoints/wgan'
}

# Define corresponding generator classes
generator_classes = {
    'DCGAN': DCGANGenerator,
    'WGAN-GP': WGANGPGenerator
}

# Configuration
num_samples_for_fid = 500  # Number of real/fake samples for FID calculation
latent_dim = 10

print(f"Evaluating {num_samples_for_fid} real and generated images for each checkpoint...")
print(f"Using device: {device}\n")

# Evaluate all checkpoints
results = evaluate_all_checkpoints(
    checkpoint_dirs=checkpoint_dirs,
    generator_classes=generator_classes,
    image_paths=image_paths,
    latent_dim=latent_dim,
    num_samples=num_samples_for_fid,
    device=device
)

Evaluating 500 real and generated images for each checkpoint...
Using device: cuda


Evaluating DCGAN checkpoints from: checkpoints/dcgan

Evaluating: epoch_1000_DCGAN_2026-03-05_18-14-10_checkpoint.pth
✗ Error evaluating epoch_1000_DCGAN_2026-03-05_18-14-10_checkpoint.pth: Error(s) in loading state_dict for DCGANGenerator:
	Missing key(s) in state_dict: "linear.weight", "linear.bias", "main.0.weight", "main.0.bias", "main.1.weight", "main.1.bias", "main.1.running_mean", "main.1.running_var", "main.3.weight", "main.3.bias", "main.4.weight", "main.4.bias", "main.4.running_mean", "main.4.running_var", "main.6.weight", "main.6.bias", "main.7.weight", "main.7.bias", "main.7.running_mean", "main.7.running_var", "main.9.weight", "main.9.bias". 
	Unexpected key(s) in state_dict: "model.0.weight", "model.0.bias", "model.3.weight", "model.3.bias", "model.4.weight", "model.4.bias", "model.4.running_mean", "model.4.running_var", "model.4.num_batches_tracked", "model.6.weight", "model.6.bias", "mo

c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\fid_evaluation.py:229: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoin

## 4. Compare Models Across Hyperparameters

In [5]:
# Convert results to a cleaner format for comparison
comparison_data = []

for model_name, model_results in results.items():
    for checkpoint_name, fid_score in model_results.items():
        if fid_score is not None:
            # Extract epoch from checkpoint name
            epoch = checkpoint_name.replace('epoch_', '').replace('_checkpoint.pth', '')
            comparison_data.append({
                'Model': model_name,
                'Checkpoint': checkpoint_name,
                'Epoch': epoch,
                'FID Score': fid_score
            })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

if len(comparison_df) > 0:
    print("\nDetailed FID Comparison:")
    print("="*70)
    print(comparison_df.to_string(index=False))
    
    # Summary statistics
    print("\n" + "="*70)
    print("Summary Statistics by Model:")
    print("="*70)
    summary = comparison_df.groupby('Model')['FID Score'].agg(['count', 'min', 'max', 'mean', 'std'])
    print(summary)
else:
    print("No results to compare. Check checkpoint paths and generator classes.")

No results to compare. Check checkpoint paths and generator classes.


## 5. Visualize FID Results

In [6]:
if len(comparison_df) > 0:
    # Create visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Bar chart comparing FID scores by model
    ax1 = axes[0]
    model_fid = comparison_df.groupby('Model')['FID Score'].mean()
    colors = ['#FF6B6B', '#4ECDC4']
    model_fid.plot(kind='bar', ax=ax1, color=colors[:len(model_fid)])
    ax1.set_title('Average FID Score by Model', fontsize=14, fontweight='bold')
    ax1.set_ylabel('FID Score (lower is better)', fontsize=12)
    ax1.set_xlabel('Model', fontsize=12)
    ax1.grid(axis='y', alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Plot 2: Line plot showing FID progression across training epochs
    ax2 = axes[1]
    for model_name in comparison_df['Model'].unique():
        model_data = comparison_df[comparison_df['Model'] == model_name].copy()
        # Convert epoch to numeric if possible
        try:
            model_data['Epoch_num'] = pd.to_numeric(model_data['Epoch'])
            model_data = model_data.sort_values('Epoch_num')
        except:
            pass
        
        ax2.plot(range(len(model_data)), model_data['FID Score'], 
                marker='o', label=model_name, linewidth=2, markersize=8)
    
    ax2.set_title('FID Score Across Training Epochs', fontsize=14, fontweight='bold')
    ax2.set_ylabel('FID Score (lower is better)', fontsize=12)
    ax2.set_xlabel('Checkpoint', fontsize=12)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Create a detailed comparison table
    print("\n" + "="*70)
    print("Detailed Checkpoint Comparison")
    print("="*70)
    
    for model_name in sorted(comparison_df['Model'].unique()):
        print(f"\n{model_name}:")
        model_df = comparison_df[comparison_df['Model'] == model_name].sort_values('FID Score')
        print(model_df[['Checkpoint', 'FID Score']].to_string(index=False))
        
        best = model_df['FID Score'].min()
        worst = model_df['FID Score'].max()
        print(f"  Best FID: {best:.4f} | Worst FID: {worst:.4f}")
else:
    print("No results to visualize.")

No results to visualize.


## 6. Analysis and Recommendations

In [7]:
if len(comparison_df) > 0:
    print("\n" + "="*70)
    print("ANALYSIS AND INSIGHTS")
    print("="*70)
    
    # Find best model overall
    best_idx = comparison_df['FID Score'].idxmin()
    best_row = comparison_df.loc[best_idx]
    print(f"\n✓ Best Model: {best_row['Model']} - {best_row['Checkpoint']}")
    print(f"  FID Score: {best_row['FID Score']:.4f}")
    
    # Analyze convergence
    print("\n📊 Convergence Analysis:")
    for model_name in sorted(comparison_df['Model'].unique()):
        model_data = comparison_df[comparison_df['Model'] == model_name]
        fid_values = model_data['FID Score'].values
        
        if len(fid_values) > 1:
            improvement = fid_values[0] - fid_values[-1]
            avg_fid = fid_values.mean()
            
            print(f"\n  {model_name}:")
            print(f"    Initial FID:  {fid_values[0]:.4f}")
            print(f"    Final FID:    {fid_values[-1]:.4f}")
            print(f"    Improvement:  {improvement:.4f} ({improvement/fid_values[0]*100:.1f}%)")
            print(f"    Average FID:  {avg_fid:.4f}")
            
            # Convergence quality
            if improvement > 0:
                print(f"    Status: ✓ Improving")
            elif improvement < 0:
                print(f"    Status: ✗ Degrading")
            else:
                print(f"    Status: ~ Stable")

    print("\n" + "="*70)
    print("INTERPRETATION GUIDE")
    print("="*70)
    print("""
FID Score Quality Levels:
  • FID < 10   : Excellent - High quality, diverse generated images
  • FID 10-20  : Good - Decent quality, minor artifacts
  • FID 20-50  : Fair - Noticeable artifacts, limited diversity
  • FID > 50   : Poor - Significant quality issues

Next Steps for Hyperparameter Tuning:
  1. Identify the best-performing checkpoint (lowest FID)
  2. Note its hyperparameters (learning rate, batch size, etc.)
  3. Try variations around those parameters
  4. Re-run FID evaluation for comparison
  5. Plot trends to identify patterns
    """)
else:
    print("Unable to perform analysis - no evaluation results available.")

Unable to perform analysis - no evaluation results available.


In [8]:
if len(comparison_df) > 0:
    # Create visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Bar chart comparing FID scores by model
    ax1 = axes[0]
    model_fid = comparison_df.groupby('Model')['FID Score'].mean()
    colors = ['#FF6B6B', '#4ECDC4']
    model_fid.plot(kind='bar', ax=ax1, color=colors[:len(model_fid)])
    ax1.set_title('Average FID Score by Model', fontsize=14, fontweight='bold')
    ax1.set_ylabel('FID Score (lower is better)', fontsize=12)
    ax1.set_xlabel('Model', fontsize=12)
    ax1.grid(axis='y', alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Plot 2: Line plot showing FID progression across training epochs
    ax2 = axes[1]
    for model_name in comparison_df['Model'].unique():
        model_data = comparison_df[comparison_df['Model'] == model_name].copy()
        # Convert epoch to numeric if possible
        try:
            model_data['Epoch_num'] = pd.to_numeric(model_data['Epoch'])
            model_data = model_data.sort_values('Epoch_num')
        except:
            pass
        
        ax2.plot(range(len(model_data)), model_data['FID Score'], 
                marker='o', label=model_name, linewidth=2, markersize=8)
    
    ax2.set_title('FID Score Across Training Epochs', fontsize=14, fontweight='bold')
    ax2.set_ylabel('FID Score (lower is better)', fontsize=12)
    ax2.set_xlabel('Checkpoint', fontsize=12)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Create a detailed comparison table
    print("\n" + "="*70)
    print("Detailed Checkpoint Comparison")
    print("="*70)
    
    for model_name in sorted(comparison_df['Model'].unique()):
        print(f"\n{model_name}:")
        model_df = comparison_df[comparison_df['Model'] == model_name].sort_values('FID Score')
        print(model_df[['Checkpoint', 'FID Score']].to_string(index=False))
        
        best = model_df['FID Score'].min()
        worst = model_df['FID Score'].max()
        print(f"  Best FID: {best:.4f} | Worst FID: {worst:.4f}")
else:
    print("No results to visualize.")

No results to visualize.
